# 01. 이산 마스킹 확산 기초

목표: clean token sequence의 각 위치를 확률 `t`로 `[MASK]`로 바꾸는 forward process를 구현합니다. Python 표준 라이브러리만 사용합니다.

In [ ]:
import random

MASK = "[MASK]"
tokens = "확산 언어 모델 은 여러 토큰 을 병렬 로 복원 한다".split()

def forward_mask(tokens, t, seed=0):
    rng = random.Random(seed)
    return [MASK if rng.random() < t else token for token in tokens]

for t in (0.0, 0.25, 0.5, 0.75, 1.0):
    noisy = forward_mask(tokens, t, seed=7)
    ratio = noisy.count(MASK) / len(noisy)
    print(f"t={t:.2f} actual_mask_ratio={ratio:.2f} ::", " ".join(noisy))

## Monte Carlo에서 평균 mask 비율 확인

Sequence가 짧으면 한 sample의 비율은 `t`와 다르지만 여러 sample 평균은 `t`에 가까워집니다.

In [ ]:
def average_mask_ratio(length, t, trials=2_000):
    total = 0
    for seed in range(trials):
        sequence = [str(i) for i in range(length)]
        total += forward_mask(sequence, t, seed).count(MASK) / length
    return total / trials

for t in (0.1, 0.5, 0.9):
    print(f"target={t:.1f}, observed={average_mask_ratio(64, t):.3f}")

## `1/t` 가중치 직관

아래 계산은 실제 model loss가 아니라, masking된 위치의 평균 loss에 `1/t`를 곱하는 구조만 보여 줍니다. 매우 작은 `t`는 weight가 크므로 수치 안정성 처리가 필요합니다.

In [ ]:
def weighted_mask_loss(token_losses, mask_flags, t):
    masked = [loss for loss, is_masked in zip(token_losses, mask_flags) if is_masked]
    return 0.0 if not masked else sum(masked) / len(masked) / t

losses = [0.2, 0.8, 0.1, 0.5]
flags = [True, False, True, False]
for t in (0.25, 0.5, 1.0):
    print(f"t={t:.2f}, weighted loss={weighted_mask_loss(losses, flags, t):.3f}")